In [11]:
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
import pylake

In [2]:
model = 'geneva_dummy_extended'
mitgcm_config, ds_to_plot = open_mitgcm_ds_from_config('..//config.json', model)

In [3]:
grid_resolution = 200
ds_to_plot['YC'] = np.arange(1, len(ds_to_plot['YC'])+1) * grid_resolution - grid_resolution/2
ds_to_plot['XC'] = np.arange(1, len(ds_to_plot['XC'])+1) * grid_resolution - grid_resolution/2
ds_to_plot['YG'] = np.arange(0, len(ds_to_plot['YG'])) * grid_resolution
ds_to_plot['XG'] = np.arange(0, len(ds_to_plot['XG'])) * grid_resolution

plt.imshow(ds_to_plot.THETA.isel(time=-1,Z=0))
plt.gca().invert_yaxis()

In [4]:
ds_to_plot['theta_nan'] = ds_to_plot['THETA'].where(ds_to_plot['THETA'] != 0, np.nan).compute()

In [5]:
ds_to_plot['mean_temp_profile'] = ds_to_plot.theta_nan.mean(dim=['XC','YC']).compute()

ds_to_plot['mean_temp_profile'].plot()

In [6]:
N_mean = pylake.buoyancy_freq(ds_to_plot['mean_temp_profile'].isel(time=-1).values, depth=ds_to_plot.Z.values, g=9.81)

def get_vertical_displacement(target_temp, ds_temperature):
    z_closest_idx = np.abs(ds_temperature - target_temp).argmin(dim='Z').values
    z_closest_value = ds_temperature['Z'].isel(Z=z_closest_idx)

    return z_closest_value

v_displacement=[]
for idx_z, mean_temp in enumerate(ds_to_plot['mean_temp_profile'].isel(time=-1).values):
    for x_idx, xc in enumerate(ds_to_plot['XC'].values):
        for y_idx, yc in enumerate(ds_to_plot['YC'].values):
            ds_temperature = ds_to_plot.THETA.isel(time=-1, XC=x_idx, YC=y_idx)
            v_displacement.append(
                {"idx_z": idx_z,
                 "x_idx": x_idx,
                 "y_idx": y_idx,
                 "v_disp": get_vertical_displacement(mean_temp, ds_temperature)
                 }
            )

v_disp = np.zeros(ds_to_plot.THETA.shape)

mean_temp_profile = ds_to_plot['mean_temp_profile']
idx_z = 10

target_temp = mean_temp_profile.isel(Z=idx_z).broadcast_like(ds_to_plot.THETA)

# Compute absolute difference
diff = np.abs(ds_to_plot.THETA - target_temp)

z_closest_idx = diff.argmin(dim='Z').compute()
z_closest = ds_to_plot['Z'].isel(Z=z_closest_idx)

ref_depth = ds_to_plot['Z'].isel(Z=idx_z)
v_disp[:,idx_z] = z_closest - ref_depth

In [34]:
mask = ds_to_plot.THETA.isel(Z=0, time=0).values != 0

In [ ]:
# Initialize v_disp with same shape as THETA
v_disp = xr.full_like(ds_to_plot.THETA, fill_value=np.nan)

for idx_z in range(ds_to_plot.sizes['Z']):
    # Reference mean profile value at this depth
    target_temp = ds_to_plot['mean_temp_profile'].isel(Z=idx_z).broadcast_like(ds_to_plot.THETA)

    # Compute difference
    diff = np.abs(ds_to_plot.THETA - target_temp)
    z_closest_idx = diff.argmin(dim='Z').compute().values  # numpy indices
    z_closest = ds_to_plot['Z'].values[z_closest_idx]  # (time, YC, XC)

    # Displacement relative to this reference depth
    ref_depth = ds_to_plot['Z'].isel(Z=idx_z).values
    v_disp.isel(Z=idx_z)[:] = np.where(mask, z_closest - ref_depth, np.nan)

array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       ...,

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan